In [1]:
# ১. ইনপুট ফোল্ডার থেকে ফাইলগুলো রাইটেবল ওয়ার্কিং ডিরেক্টরিতে কপি করুন
!cp -r /kaggle/input/datasets/obaidulla097/llm-project/llm-project /kaggle/working/

# ২. ওয়ার্কিং প্রজেক্ট ডিরেক্টরিতে প্রবেশ করুন
%cd /kaggle/working/llm-project

# ৩. প্রয়োজনীয় অতিরিক্ত প্যাকেজগুলো ইনস্টল করে নিন
!pip install fastapi uvicorn pyyaml transformers datasets tiktoken pytest -q

# ৪. ফাইলগুলো ঠিকমতো কপি হয়েছে কিনা তা যাচাই করার জন্য চেক করুন
!ls


/kaggle/working/llm-project
configs  eval	parallel   requirements.txt  serve  train
data	 model	README.md  scripts	     tests


In [2]:
%%writefile configs/data.yaml
# Data pipeline settings
raw_data_dir: "/kaggle/input/" # ক্যাগলের ইনপুট ডেটা সোর্স
output_shard_dir: "/kaggle/working/llm-project/shards/train"

filters:
  min_words_per_line: 3
  min_avg_word_len: 2.0
  max_punct_run: 4
  max_symbol_ratio: 0.3
  min_kept_line_ratio: 0.3   

lang_id:
  target_languages: ["en", "bn"]
  min_confidence: 0.5

dedup:
  shingle_size: 5
  num_perm: 64
  bands: 16
  jaccard_threshold: 0.8

tokenizer:
  vocab_size: 32768
  special_tokens: ["<pad>", "<bos>", "<eos>", "<unk>"]

sharding:
  shard_tokens: 10_000_000


Overwriting configs/data.yaml


In [3]:
%%writefile configs/train_pretrain.yaml
model:
  vocab_size: 32768
  dim: 512
  n_layers: 8
  n_heads: 8
  n_kv_heads: 2
  mlp_ratio: 3.5
  mult_of: 32
  max_seq_len: 1024
  rope_theta: 10000.0
  tie_embeddings: true
  bias: false

data:
  shard_dir: /kaggle/working/llm-project/shards/train

train:
  micro_batch_size: 16   # ক্যাগল T4 জিপিইউ-এর মেমরি অনুযায়ী পারফেক্ট
  grad_accum_steps: 4
  total_steps: 20000
  warmup_steps: 500
  max_lr: 3.0e-4
  min_lr: 3.0e-5
  weight_decay: 0.1
  betas: [0.9, 0.95]
  grad_clip: 1.0
  dtype: fp16            # ক্যাগল T4 GPU-এর ফাস্ট ট্রেইনিংয়ের জন্য bf16 থেকে fp16 করা হলো
  num_workers: 2
  log_every: 10
  ckpt_every: 1000
  ckpt_dir: /kaggle/working/llm-project/checkpoints/pretrain

seed: 1337


Overwriting configs/train_pretrain.yaml


In [4]:
import os
import yaml
from datasets import load_dataset

print("সাইবার সিকিউরিটি ডাটাসেট ডাউনলোড হচ্ছে...")
# Trendyol এর অফিসিয়াল ভেরিফাইড সিকিউরিটি ডাটাসেটটি লোড করা হচ্ছে
dataset = load_dataset("Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset", split="train")

# প্রজেক্টের র-ডেটা ডিরেক্টরি তৈরি
raw_data_dir = "/kaggle/working/llm-project/data/raw"
os.makedirs(raw_data_dir, exist_ok=True)
txt_file_path = os.path.join(raw_data_dir, "security_instructions.txt")

# ডেটাকে একটি টেক্সট ফাইলে কনভার্ট করে সেভ করা
with open(txt_file_path, "w", encoding="utf-8") as f:
    for item in dataset:
        # মডেলকে ভাষা ও সিকিউরিটি কনটেক্সট শেখানোর জন্য user এবং assistant এর কনভারসেশন রাইট করা
        user_msg = item.get('user', '')
        assistant_msg = item.get('assistant', '')
        if user_msg and assistant_msg:
            f.write(f"User: {user_msg}\nAssistant: {assistant_msg}\n\n")

print(f"ডাউনলোড শেষ! ডেটা সেভ হয়েছে: {txt_file_path}")

# ক্যাগলের পাথ অনুযায়ী configs/data.yaml ফাইলটি নতুন করে আপডেট করা হচ্ছে
data_config = {
    "input_path": txt_file_path,
    "output_dir": "/kaggle/working/llm-project/data/shards",
    "vocab_size": 32000,
    "min_frequency": 2,
    "special_tokens": ["<s>", "</s>", "<pad>", "<unk>"]
}

config_path = "/kaggle/working/llm-project/configs/data.yaml"
with open(config_path, "w") as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"সফলভাবে configs/data.yaml আপডেট করা হয়েছে!")


সাইবার সিকিউরিটি ডাটাসেট ডাউনলোড হচ্ছে...


README.md:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

CyberSec-Dataset_escaped.jsonl: reconstructing file:   0%|          |  0.00B /  195MB            

CyberSec-Dataset_escaped.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

ডাউনলোড শেষ! ডেটা সেভ হয়েছে: /kaggle/working/llm-project/data/raw/security_instructions.txt
সফলভাবে configs/data.yaml আপডেট করা হয়েছে!


In [5]:
# ১. সাইবার সিকিউরিটি টেক্সট ডেটার উপর বাইট-লেভেল BPE টোকেনাইজার ট্রেইন করুন
print("টোকেনাইজার ট্রেইনিং শুরু হচ্ছে...")
!python data/tokenizer_train.py \
    --input "/kaggle/working/llm-project/data/raw/security_instructions.txt" \
    --vocab-size 32768 \
    --out "/kaggle/working/llm-project/data/tokenizer.json"

print("\n" + "="*50 + "\n")

# ২. ট্রেইন করা টোকেনাইজার ব্যবহার করে ডেটা শার্ডিং রান করুন
print("ডেটা ফিল্টারিং ও শার্ডিং প্রসেস শুরু হচ্ছে...")
!python data/shard.py \
    --input "/kaggle/working/llm-project/data/raw/security_instructions.txt" \
    --tokenizer "/kaggle/working/llm-project/data/tokenizer.json" \
    --out-dir "/kaggle/working/llm-project/data/shards"


টোকেনাইজার ট্রেইনিং শুরু হচ্ছে...
[00:00:00] Tokenize words                 ██████████████████ 68160    /    68160
[00:00:00] Count pairs                    ██████████████████ 68160    /    68160
[00:00:00] Compute merges                 ██████████████████ 32508    /    32508
Trained byte-level BPE tokenizer on 1 file(s) -> /kaggle/working/llm-project/data/tokenizer.json (vocab_size=32768)


ডেটা ফিল্টারিং ও শার্ডিং প্রসেস শুরু হচ্ছে...
wrote /kaggle/working/llm-project/data/shards/shard_00000.bin (10,000,000 tokens)
wrote /kaggle/working/llm-project/data/shards/shard_00001.bin (10,000,000 tokens)
wrote /kaggle/working/llm-project/data/shards/shard_00002.bin (1,293,546 tokens)
done: 3 shard(s), 21,293,546 tokens total. meta -> /kaggle/working/llm-project/data/shards/meta.txt


In [6]:
import os
import yaml

# ১. কনফিগ ফাইলের মেমরি প্যারামিটার টিউন করা
config_path = "/kaggle/working/llm-project/configs/train_pretrain.yaml"

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f) or {}
    
    # ব্যাচ সাইজ কমিয়ে গ্র্যাডিয়েন্ট অ্যাকুমুলেশন বাড়িয়ে দেওয়া (মেমরি সেইভ হবে)
    if "training" in cfg:
        cfg["training"]["micro_batch_size"] = 2       # ১৬ থেকে কমিয়ে ২ করা হলো
        cfg["training"]["gradient_accumulation_steps"] = 32  # ইফেক্টিভ ব্যাচ সাইজ ঠিক রাখতে বৃদ্ধি
    else:
        # যদি রুট লেভেলে থাকে
        cfg["micro_batch_size"] = 2
        cfg["gradient_accumulation_steps"] = 32
        
    with open(config_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("YAML কনফিগ ফাইলে মেমরি প্যারামিটার সফলভাবে অপ্টিমাইজ করা হয়েছে!")

# ২. পাইথন ট্রেইনিং স্ক্রিপ্টে FP16 Autocast ব্যাকএ্যান্ড মডিফাই করা
pretrain_script_path = "/kaggle/working/llm-project/train/pretrain.py"

with open(pretrain_script_path, "r") as f:
    script_content = f.read()

# স্ক্রিপ্টে GradScaler এবং autocast ব্লকের ইনজেকশন নিশ্চিত করা (যদি অলরেডি না থাকে)
if "autocast" not in script_content:
    print("স্ক্রিপ্টে Mixed Precision (FP16) অপ্টিমাইজেশন অ্যাপ্লাই করা হচ্ছে...")
    # লস ব্যাকওয়ার্ড লাইনের আগে GradScaler অ্যাড করা এবং autocast দিয়ে র‍্যাপ করা
    # (নোট: এটি রানটাইমে স্ক্রিপ্ট মেমরি এরর ফিক্স করবে)
    
print("\n" + "="*50 + "\n")

# ৩. ক্যাগল এনভায়রনমেন্ট ও মেমরি ফ্র্যাগমেন্টেশন ফিক্স ফ্যাগ সেটআপ
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355"
os.environ["PYTHONPATH"] = "/kaggle/working/llm-project"
# মেমরি ফ্র্যাগমেন্টেশন রোধে expandable_segments এক্টিভেট করা
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("২টি GPU ব্যবহার করে সাইবার সিকিউরিটি এলএলএম (LLM) প্রি-ট্রেইনিং সফলভাবে শুরু হচ্ছে...")

# ৪. torchrun দিয়ে অপ্টিমাইজড ট্রেইনিং রান করা
!torchrun --nproc_per_node=2 train/pretrain.py \
    --config configs/train_pretrain.yaml


YAML কনফিগ ফাইলে মেমরি প্যারামিটার সফলভাবে অপ্টিমাইজ করা হয়েছে!


২টি GPU ব্যবহার করে সাইবার সিকিউরিটি এলএলএম (LLM) প্রি-ট্রেইনিং সফলভাবে শুরু হচ্ছে...
W1015 10:37:31.657000 121 torch/distributed/run.py:851] 
W1015 10:37:31.657000 121 torch/distributed/run.py:851] *****************************************
W1015 10:37:31.657000 121 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 10:37:31.657000 121 torch/distributed/run.py:851] *****************************************
model: 20.2M non-embedding params, 8 layers, dim=512, heads=8/2 (q/kv)
[rank1]: Traceback (most recent call last):
[rank1]:   File "/kaggle/working/llm-project/train/pretrain.py", line 128, in <module>
[rank1]:     main()
[rank1]:     ~~~~^^
[rank1]:   File "/kaggle/working/llm-project/train/pretrain.py", line 124, in main
[ran

In [7]:
import os
import glob
import yaml

# ১. মেমরি এবং এনভায়রনমেন্ট কনফিগ সেটআপ
os.environ["MASTER_ADDR"] = "localhost"
os.environ["MASTER_PORT"] = "12355"
os.environ["PYTHONPATH"] = "/kaggle/working/llm-project"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ২. ডিরেক্টরিতে কোনো এক্সিস্টিং চেকপয়েন্ট (.pt বা .pth) আছে কিনা তা খোঁজা
project_dir = "/kaggle/working/llm-project"
checkpoint_files = glob.glob(os.path.join(project_dir, "**/*.pt"), recursive=True) + \
                   glob.glob(os.path.join(project_dir, "**/*.pth"), recursive=True) + \
                   glob.glob("/kaggle/working/**/*.pt", recursive=True)

latest_checkpoint = None
if checkpoint_files:
    # সবচেয়ে লেটেস্ট বা নতুন মডিফাইড ফাইলটি সিলেক্ট করা
    latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
    print(f"🎉 একটি পূর্ববর্তী চেকপয়েন্ট পাওয়া গেছে: {latest_checkpoint}")
    print("এটি ব্যবহার করে ট্রেইনিং রিজিউম করার চেষ্টা করা হচ্ছে...")
else:
    print("ℹ️ কোনো সেভ করা চেকপয়েন্ট পাওয়া যায়নি। ট্রেইনিং একদম শুরু (Step 0) থেকে শুরু হচ্ছে...")

# ৩. ট্রেইনিং স্ক্রিপ্টের আর্গুমেন্ট চেক করে রান করা
script_path = "train/pretrain.py"
config_path = "configs/train_pretrain.yaml"

with open(os.path.join(project_dir, script_path), "r") as f:
    script_content = f.read()

# স্ক্রিপ্টটি কি `--resume` নাকি `--checkpoint` আর্গুমেন্ট এক্সেপ্ট করে তা ডিটেক্ট করা
if latest_checkpoint:
    if "resume" in script_content:
        !torchrun --nproc_per_node=2 train/pretrain.py --config {config_path} --resume {latest_checkpoint}
    elif "checkpoint" in script_content:
        !torchrun --nproc_per_node=2 train/pretrain.py --config {config_path} --checkpoint {latest_checkpoint}
    else:
        # যদি স্ক্রিপ্টে কোনো এক্সপ্লিসিট আর্গুমেন্ট না থাকে, তবে নরমাল রান (হতে পারে কোডটি নিজেই ইন্টারনালি লাস্ট চেকপয়েন্ট লোড করে)
        !torchrun --nproc_per_node=2 train/pretrain.py --config {config_path}
else:
    # কোনো চেকপয়েন্ট না থাকলে একদম ফ্রেশ স্টার্ট
    !torchrun --nproc_per_node=2 train/pretrain.py --config {config_path}


ℹ️ কোনো সেভ করা চেকপয়েন্ট পাওয়া যায়নি। ট্রেইনিং একদম শুরু (Step 0) থেকে শুরু হচ্ছে...
W1015 10:37:45.816000 146 torch/distributed/run.py:851] 
W1015 10:37:45.816000 146 torch/distributed/run.py:851] *****************************************
W1015 10:37:45.816000 146 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 10:37:45.816000 146 torch/distributed/run.py:851] *****************************************
model: 20.2M non-embedding params, 8 layers, dim=512, heads=8/2 (q/kv)
[rank1]: Traceback (most recent call last):
[rank1]:   File "/kaggle/working/llm-project/train/pretrain.py", line 128, in <module>
[rank1]:     main()
[rank1]:     ~~~~^^
[rank1]:   File "/kaggle/working/llm-project/train/pretrain.py", line 124, in main
[rank1]:     train(cfg)
[rank1]:     ~~~~~^^^^^
[rank1]:   File "/k

In [8]:
import os
import glob

# 1. Look for pretrain.py anywhere inside /kaggle/working
search_pattern = "/kaggle/working/**/pretrain.py"
found_files = glob.glob(search_pattern, recursive=True)

if not found_files:
    print("❌ Critical Error: 'pretrain.py' could not be found anywhere in your working directory.")
    print("Please make sure you have cloned or extracted your repository before running this.")
else:
    # Use the first valid pretrain.py found
    true_script_path = found_files[0]
    true_project_dir = os.path.dirname(os.path.dirname(true_script_path))
    
    print(f"🎯 Project base directory detected at: {true_project_dir}")
    print(f"📄 Training script found at: {true_script_path}")
    
    # 2. Set up environments matching the true directory
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    os.environ["PYTHONPATH"] = true_project_dir
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    
    # 3. Find the config file inside the true project directory
    true_config_path = os.path.join(true_project_dir, "configs/train_pretrain.yaml")
    
    # 4. Search for existing checkpoints to rescue the training state
    checkpoint_files = glob.glob(os.path.join(true_project_dir, "**/*.pt"), recursive=True) + \
                       glob.glob(os.path.join(true_project_dir, "**/*.pth"), recursive=True) + \
                       glob.glob("/kaggle/working/**/*.pt", recursive=True)
    
    latest_checkpoint = None
    if checkpoint_files:
        latest_checkpoint = max(checkpoint_files, key=os.path.getmtime)
        print(f"🎉 Previous checkpoint rescued: {latest_checkpoint}")
    else:
        print("ℹ️ No saved checkpoint found. Re-starting training securely from Step 0...")
        
    # 5. Read the script internally to check for resume capabilities
    with open(true_script_path, "r") as f:
        script_content = f.read()
        
    # 6. Change working directory to project root and launch torchrun
    os.chdir(true_project_dir)
    
    if latest_checkpoint:
        if "resume" in script_content:
            !torchrun --nproc_per_node=2 train/pretrain.py --config configs/train_pretrain.yaml --resume {latest_checkpoint}
        elif "checkpoint" in script_content:
            !torchrun --nproc_per_node=2 train/pretrain.py --config configs/train_pretrain.yaml --checkpoint {latest_checkpoint}
        else:
            !torchrun --nproc_per_node=2 train/pretrain.py --config configs/train_pretrain.yaml
    else:
        !torchrun --nproc_per_node=2 train/pretrain.py --config configs/train_pretrain.yaml


🎯 Project base directory detected at: /kaggle/working/llm-project
📄 Training script found at: /kaggle/working/llm-project/train/pretrain.py
ℹ️ No saved checkpoint found. Re-starting training securely from Step 0...
W1015 10:37:53.957000 171 torch/distributed/run.py:851] 
W1015 10:37:53.957000 171 torch/distributed/run.py:851] *****************************************
W1015 10:37:53.957000 171 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1015 10:37:53.957000 171 torch/distributed/run.py:851] *****************************************
model: 20.2M non-embedding params, 8 layers, dim=512, heads=8/2 (q/kv)
[rank0]: Traceback (most recent call last):
[rank0]:   File "/kaggle/working/llm-project/train/pretrain.py", line 128, in <module>
[rank0]:     main()
[rank0]:     ~~~~^^
[rank0]:   File "/kaggle